# Spectral Decomposition

Decompose an image into wavelength bands using the **CIE 1931 colour matching functions**, apply per-band radial offsets based on glass dispersion, then recompose into sRGB. This gives more realistic chromatic aberration than simple RGB channel shifting because it accounts for the *continuous* nature of the spectrum.

**Why this matters**: Simple RGB shifting creates a 3-colour fringe (red, empty gap, blue). Real lenses create a continuous spectral smear that includes cyan, yellow, and violet intermediates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from PIL import Image

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. CIE Colour Matching Functions (Gaussian Approximation)

In [ ]:
def cie_cmf_approx(wavelength_nm: np.ndarray):
    """
    Simplified CIE 1931 2° CMFs using Gaussian lobes.
    Returns (x_bar, y_bar, z_bar) arrays.
    Accurate enough for visualisation; use tabulated data for metrology.
    """
    def gauss(lam, mu, sigma, A):
        return A * np.exp(-0.5 * ((lam - mu) / sigma) ** 2)

    x_bar = (gauss(wavelength_nm, 600, 35, 1.056) +
             gauss(wavelength_nm, 446, 19, 0.362) -
             gauss(wavelength_nm, 556, 24, 0.065))
    y_bar =  gauss(wavelength_nm, 556, 42, 0.821) + gauss(wavelength_nm, 448, 24, 0.286)
    z_bar = (gauss(wavelength_nm, 451, 28, 1.217) + gauss(wavelength_nm, 374, 15, 0.013))

    return np.clip(x_bar, 0, None), np.clip(y_bar, 0, None), np.clip(z_bar, 0, None)

# Spectral bands: 7 samples from 400 to 700 nm
BANDS = np.linspace(400, 700, 7)
x_bar, y_bar, z_bar = cie_cmf_approx(BANDS)

# Build XYZ → sRGB matrix (D65 illuminant, sRGB primaries)
XYZ_TO_RGB = np.array([
    [ 3.2404542, -1.5371385, -0.4985314],
    [-0.9692660,  1.8760108,  0.0415560],
    [ 0.0556434, -0.2040259,  1.0572252],
])

# Precompute the weight each band contributes to sRGB
xyz_weights = np.stack([x_bar, y_bar, z_bar], axis=1)   # (7, 3)
band_to_rgb = xyz_weights @ XYZ_TO_RGB.T                # (7, 3) — each row is sRGB weight

# Normalise so the white (equal energy) reconstructs to white
white = band_to_rgb.sum(axis=0)
band_to_rgb /= white[np.newaxis, :]

# Plot CMFs and band positions
lam_cont = np.linspace(380, 750, 400)
xc, yc, zc = cie_cmf_approx(lam_cont)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(lam_cont, xc, 'r', lw=1.5, label='x̄(λ)')
ax.plot(lam_cont, yc, 'g', lw=1.5, label='ȳ(λ)')
ax.plot(lam_cont, zc, 'b', lw=1.5, label='z̄(λ)')
ax.vlines(BANDS, 0, 1, colors='gray', lw=0.8, ls='--', label='Sample bands')
ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('CMF value')
ax.set_title('CIE 1931 colour matching functions (Gaussian approx.) + spectral bands')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Dispersion-Based Band Shifts

Using a Cauchy-type dispersion model, we compute the radial shift for each wavelength relative to the green reference.

In [ ]:
def band_shifts(wavelengths_nm, abbe_v=40.0, dispersion_strength=0.012):
    """
    Compute per-band radial shift coefficient (fraction of image radius).
    dispersion_strength : max shift at the image edge (in UV coordinates).
    """
    lam_green = 546.1  # reference
    # Hartmann-style relative shift: proportional to 1/λ² deviation from green
    shifts = (1.0 / (wavelengths_nm / lam_green) ** 2 - 1.0) / abbe_v
    # Normalise so the peak absolute shift equals dispersion_strength
    shifts *= dispersion_strength / max(abs(shifts))
    return shifts

shifts_bk7  = band_shifts(BANDS, abbe_v=64, dispersion_strength=0.008)
shifts_sf11 = band_shifts(BANDS, abbe_v=26, dispersion_strength=0.008)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.stem(BANDS, shifts_bk7,  linefmt='b-', markerfmt='bs', basefmt=' ', label='N-BK7 (V=64)')
ax.stem(BANDS + 5, shifts_sf11, linefmt='r-', markerfmt='r^', basefmt=' ', label='N-SF11 (V=26)')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Radial shift coefficient')
ax.set_title('Per-band dispersion shift')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Apply Spectral CA to a Test Image

We'll generate a synthetic test image (white circle on black = artificial star field) and apply spectral CA.

In [ ]:
def make_test_image(size=400):
    """White point sources on black background — ideal for CA visualisation."""
    img = np.zeros((size, size, 3), dtype=np.float32)
    cx, cy = size // 2, size // 2
    # Central star
    img[cy-2:cy+3, cx-2:cx+3] = 1.0
    # Off-axis stars
    for ox, oy in [(120, 120), (260, 120), (120, 260), (260, 260),
                   (50, 200), (350, 200), (200, 50), (200, 350)]:
        img[oy-2:oy+3, ox-2:ox+3] = 1.0
    return img

def apply_spectral_ca(image_rgb, abbe_v=40, dispersion=0.015):
    """
    Apply spectral chromatic aberration using 7-band CIE decomposition.
    image_rgb : float32 (H, W, 3) in [0, 1]
    """
    H, W, _ = image_rgb.shape
    shifts = band_shifts(BANDS, abbe_v=abbe_v, dispersion_strength=dispersion)

    # Precompute center-relative grid
    cx, cy = W / 2.0, H / 2.0
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    dx = (xs - cx) / (W / 2.0)  # normalised to [-1, 1]
    dy = (ys - cy) / (H / 2.0)

    # Luminance of the source image (treat it as a flat spectrum)
    lum = image_rgb @ np.array([0.2126, 0.7152, 0.0722])

    output = np.zeros((H, W, 3), dtype=np.float32)

    for band_idx, (shift, w_rgb) in enumerate(zip(shifts, band_to_rgb)):
        # Shifted sampling coordinates
        src_row = cy + (dy + dy * shift) * (H / 2.0)
        src_col = cx + (dx + dx * shift) * (W / 2.0)

        sampled = map_coordinates(lum, [src_row, src_col], order=1, mode='constant')

        # Accumulate into sRGB channels
        for c in range(3):
            output[:, :, c] += sampled * w_rgb[c]

    return np.clip(output, 0, 1)

# Generate and process
original = make_test_image(400)
ca_bk7   = apply_spectral_ca(original, abbe_v=64, dispersion=0.02)
ca_sf11  = apply_spectral_ca(original, abbe_v=26, dispersion=0.02)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, img, title in zip(axes,
                           [original, ca_bk7, ca_sf11],
                           ['Original', 'Spectral CA — N-BK7 (V=64)', 'Spectral CA — N-SF11 (V=26)']):
    ax.imshow(np.clip(img, 0, 1), origin='upper')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Chromatic Fringe Analysis

Inspect a horizontal cross-section through an off-axis star to see the spectral colour distribution.

In [ ]:
# Cross-section at y=120 (through the top-right off-axis star)
y_slice = 120
x = np.arange(400)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for ax, img, title, ls in zip(axes, [ca_bk7, ca_sf11],
                               ['N-BK7 (V=64) — low dispersion', 'N-SF11 (V=26) — high dispersion'],
                               ['-', '-']):
    ax.plot(x, img[y_slice, :, 0], 'r', lw=1.5, label='R', ls=ls)
    ax.plot(x, img[y_slice, :, 1], 'g', lw=1.5, label='G', ls=ls)
    ax.plot(x, img[y_slice, :, 2], 'b', lw=1.5, label='B', ls=ls)
    ax.set_ylabel('Intensity')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(100, 160)

axes[1].set_xlabel('Pixel position (x)')
plt.suptitle('Chromatic fringe cross-section through off-axis star', y=1.02)
plt.tight_layout()
plt.show()